In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("../data/FULL_INSECT_DATA.csv")
df.head()

,basisOfRecord,eventDate,eventTime,day,decimalLatitude,decimalLongitude,coordinateUncertaintyInMeters,order,elevation,hasCoordinate,...,land_cover_code,land_cover_label,hourly_temp_C,dewpoint_C,soil_temp_C,cloud_cover_pct,snow_depth_m,precip_m,hourly_humidity_percent,precip_type
0,HUMAN_OBSERVATION,2024-01-08 09:41:10,09:41:10-05:00,8.0,46.232197,-67.793870,34.0,Hemiptera,NaN,True,...,22,"Developed, Low Intensity",-11.906769,-14.569794,-10.137787,96.188354,0.008718,0.000000,80.564410,none
1,HUMAN_OBSERVATION,2024-01-24 17:54:00,17:54:00-05:00,24.0,43.022520,-77.060684,15.0,Lepidoptera,NaN,True,...,21,"Developed, Open Space",2.642975,0.755096,-1.829926,100.000000,0.008189,0.000518,87.362220,rain
2,HUMAN_OBSERVATION,2024-02-09 08:50:00,08:50:00-05:00,9.0,42.450188,-71.120190,6.0,Coleoptera,NaN,True,...,41,Deciduous Forest,-1.682465,-2.708221,-0.514252,67.944336,0.000000,0.000000,92.700455,none
3,HUMAN_OBSERVATION,2024-03-01 17:30:54,17:30:54-05:00,1.0,44.662766,-74.997230,23.0,Coleoptera,NaN,True,...,23,"Developed, Medium Intensity",2.897858,-7.541443,-2.525726,60.974120,0.003399,0.000000,46.182762,none
4,HUMAN_OBSERVATION,2024-03-14 13:56:54,13:56:54-04:00,14.0,43.083748,-71.976890,6.0,Lepidoptera,NaN,True,...,21,"Developed, Open Space",5.096100,1.256012,0.639557,93.933105,0.000000,0.000000,76.219760,none


In [ ]:
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import top_k_accuracy_score

df["day_of_year"] = pd.to_datetime(df["eventDate"]).dt.dayofyear
df["hour"] = pd.to_datetime(df["eventDate"]).dt.hour

# ---- 2. Filter to species with enough observations to be learnable ----
species_counts = df["species"].value_counts()
viable_species = species_counts[species_counts >= 200].index
df_viable = df[df["species"].isin(viable_species)].copy()
print(f"{len(viable_species)} species kept, {len(df_viable)} rows")

# ---- 3. Encode the categorical features ----
eco_encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
df_viable["ecoregion_enc"] = eco_encoder.fit_transform(df_viable[["ecoregion"]])

precip_encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
df_viable["precip_type_enc"] = precip_encoder.fit_transform(df_viable[["precip_type"]])

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(df_viable["species"])


df_viable["day_sin"] = np.sin(2 * np.pi * df_viable["day_of_year"] / 365.25)
df_viable["day_cos"] = np.cos(2 * np.pi * df_viable["day_of_year"] / 365.25)

df_viable["hour_sin"] = np.sin(2 * np.pi * df_viable["hour"] / 24.0)
df_viable["hour_cos"] = np.cos(2 * np.pi * df_viable["hour"] / 24.0)

# ---- 4. Final feature set — raw lat/lon are gone, replaced by everything we built ----
params = [
    "hourly_temp_C", "hourly_humidity_percent", "soil_temp_C",
    "cloud_cover_pct", "snow_depth_m", "precip_type_enc",
    "hour_sin", "hour_cos", "day_sin", "day_cos",
    "ecoregion_enc", "land_cover_code"
]
X = df_viable[params]

# ---- 5. Group-based split — same site can never appear in both train and test ----
groups = (df_viable["decimalLatitude"].round(3).astype(str)
          + "_" + df_viable["decimalLongitude"].round(3).astype(str))
gss = GroupShuffleSplit(test_size=0.2, random_state=67)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

# ---- 6. Train — same hyperparameters as your original NJ model, on purpose ----
model = RandomForestClassifier(
    n_estimators=100,
    min_samples_leaf=10,
    max_features="sqrt",
    oob_score=False,
    n_jobs=2,          # just parallelism for speed, doesn't change what gets learned
    random_state=67
)
model.fit(X_train, y_train)




batch_size = 5000
total_samples = X_train.shape[0]
correct_predictions = 0

print(f"Evaluating training accuracy across {total_samples} samples...")

# Loop through X_train in small steps
for i in range(0, total_samples, batch_size):
    X_batch = X_train[i : i + batch_size]
    y_batch = y_train[i : i + batch_size]

    # Predict for this chunk only
    batch_preds = model.predict(X_batch)

    # Count matching classifications
    correct_predictions += np.sum(batch_preds == y_batch)

# Calculate final accuracy percentage
train_accuracy = (correct_predictions / total_samples) * 100
print(f"\nSuccess! Training Accuracy: {train_accuracy:.2f}%")

print(f"Test accuracy:  {model.score(X_test, y_test):.4f}")

test_probs = model.predict_proba(X_test)
top20_acc = top_k_accuracy_score(y_test, test_probs, k=20, labels=model.classes_)
print(f"Top-20 test accuracy: {top20_acc:.4f}")

importances = pd.Series(model.feature_importances_, index=params).sort_values(ascending=False)
print(importances)


705 species kept, 626116 rows


,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",10
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",2
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",67
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total

In [8]:
batch_size = 5000
total_samples = X_train.shape[0]
correct_predictions = 0

print(f"Evaluating training accuracy across {total_samples} samples...")

# Loop through X_train in small steps
for i in range(0, total_samples, batch_size):
    X_batch = X_train[i : i + batch_size]
    y_batch = y_train[i : i + batch_size]

    # Predict for this chunk only
    batch_preds = model.predict(X_batch)

    # Count matching classifications
    correct_predictions += np.sum(batch_preds == y_batch)

# Calculate final accuracy percentage
train_accuracy = (correct_predictions / total_samples) * 100
print(f"\nSuccess! Training Accuracy: {train_accuracy:.2f}%")

print(f"Test accuracy:  {model.score(X_test, y_test):.4f}")

test_probs = model.predict_proba(X_test)
top20_acc = top_k_accuracy_score(y_test, test_probs, k=20, labels=model.classes_)
print(f"Top-20 test accuracy: {top20_acc:.4f}")

importances = pd.Series(model.feature_importances_, index=params).sort_values(ascending=False)
print(importances)

Evaluating training accuracy across 504591 samples...

Success! Training Accuracy: 48.20%
Test accuracy:  0.0725
Top-20 test accuracy: 0.4021
soil_temp_C                0.138909
hourly_humidity_percent    0.133131
hourly_temp_C              0.131636
cloud_cover_pct            0.114146
day_sin                    0.095002
day_cos                    0.090827
ecoregion_enc              0.087225
land_cover_code            0.069165
hour_cos                   0.061712
hour_sin                   0.061056
precip_type_enc            0.015792
snow_depth_m               0.001399
dtype: float64


In [9]:
import joblib

joblib.dump(model, "rf_model.pkl")

joblib.dump(eco_encoder, "eco_encoder.pkl")
joblib.dump(precip_encoder, "precip_encoder.pkl")
joblib.dump(label_encoder, "label_encoder.pkl")

['label_encoder.pkl']

lightgbm model:

In [2]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import time
import joblib
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import GroupShuffleSplit, train_test_split

# ---- 1. Load your fully-joined dataframe ----
df = pd.read_csv("../data/FULL_INSECT_DATA.csv")

df["day_of_year"] = pd.to_datetime(df["eventDate"]).dt.dayofyear
df["hour"] = pd.to_datetime(df["eventDate"]).dt.hour
df["day_sin"] = np.sin(2 * np.pi * df["day_of_year"] / 365.25)
df["day_cos"] = np.cos(2 * np.pi * df["day_of_year"] / 365.25)

df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24.0)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24.0)


# ---- 2. Build your location grouping key once, reuse everywhere ----
df["location_group"] = (df["decimalLatitude"].round(3).astype(str)
                         + "_" + df["decimalLongitude"].round(3).astype(str))

# ---- 3. Filter species TWO ways, not just by raw count ----
# (a) overall observation count — same as before
species_counts = df["species"].value_counts()
viable_by_count = species_counts[species_counts >= 200].index

# (b) geographic spread — this is the one that actually fixes the "-inf" warnings,
# since a species clustered at 2 sites has nothing to learn under a location-based split
species_spread = df.groupby("species")["location_group"].nunique()
viable_by_spread = species_spread[species_spread >= 10].index

viable_species = set(viable_by_count) & set(viable_by_spread)
df_viable = df[df["species"].isin(viable_species)].copy()
print(f"{len(viable_species)} species kept (down from {df['species'].nunique()}), {len(df_viable)} rows")

# ---- 4. Encode label and categoricals ----
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(df_viable["species"])

for col in ["ecoregion", "land_cover_code", "precip_type"]:
    df_viable[col] = df_viable[col].astype("category")

params = ["hourly_temp_C", "hourly_humidity_percent", "soil_temp_C",
          "cloud_cover_pct", "snow_depth_m", "precip_type",
          "hour_sin", "hour_cos", "day_sin", "day_cos",
          "ecoregion", "land_cover_code"]
X = df_viable[params]
cat_features = ["precip_type", "ecoregion", "land_cover_code"]

# ---- 5. Double Group-Based Split (No Spatial Leakage, Safe Evaluation) ----
# First, isolate the true test set
gss_test = GroupShuffleSplit(test_size=0.2, random_state=67)
train_idx, test_idx = next(gss_test.split(X, y, groups=df_viable["location_group"]))

X_train_full, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train_full, y_test = y[train_idx], y[test_idx]
groups_train_full = df_viable["location_group"].iloc[train_idx]

# Second, isolate a clean validation set from the training data using the same spatial logic
gss_val = GroupShuffleSplit(test_size=0.1, random_state=67)
tr_idx, val_idx = next(gss_val.split(X_train_full, y_train_full, groups=groups_train_full))

X_tr, X_val = X_train_full.iloc[tr_idx], X_train_full.iloc[val_idx]
y_tr, y_val = y_train_full[tr_idx], y_train_full[val_idx]


# ---- 6. Always calibrate timing with your actual hyperparams ----
# Added learning_rate, adjusted min_child_samples for 705 classes
calibration_model = lgb.LGBMClassifier(
    objective="multiclass", 
    num_class=len(label_encoder.classes_),
    learning_rate=0.03,        # Slower, intentional updates
    n_estimators=10, 
    num_leaves=63, 
    min_child_samples=50,      # Prevent overfitting on rare species
    verbose=-1, 
    n_jobs=-1, 
    random_state=67
)
start = time.time()
calibration_model.fit(X_tr, y_tr, categorical_feature=cat_features)
elapsed = time.time() - start
print(f"{elapsed:.1f}s for 10 rounds -> roughly {elapsed * 30:.0f}s for 300 rounds")


# ---- 7. The Real Run with Early Stopping ----
model = lgb.LGBMClassifier(
    objective="multiclassova",      # Often stabilizes high-cardinality targets
    num_class=len(label_encoder.classes_),
    learning_rate=0.03,            # Keep it steady
    n_estimators=500,              # Give it room to run now that it's regularized
    
    # Severe Overfitting Countermeasures
    max_depth=6,                   # Shallow trees force generalization
    num_leaves=31,                 # Drop from 63 to 31 to match depth cap
    min_child_samples=100,         # Require at least 100 samples per leaf
    
    # Feature & Row Subsampling (Crucial for spatial data)
    colsample_bytree=0.5,          # Select 50% of columns per tree
    subsample=0.7,                 # Row subsampling
    bagging_freq=1,                # Perform bagging at every iteration
    
    # Penalization
    reg_lambda=10.0,               # L2 regularization on weights
    
    verbose=-1, 
    n_jobs=-1, 
    random_state=67
)

model.fit(
    X_tr, y_tr, 
    eval_set=[(X_val, y_val)],
    categorical_feature=cat_features,
    callbacks=[lgb.early_stopping(stopping_rounds=20, verbose=True)] 
)

print(f"Stopped at round: {model.best_iteration_}")
print(f"Train accuracy: {model.score(X_tr, y_tr):.4f}")
print(f"Test accuracy:  {model.score(X_test, y_test):.4f}")

def chunked_top_k_accuracy(model, X_test, y_test, k=20, chunk_size=20_000):
    correct = 0
    for start in range(0, len(X_test), chunk_size):
        end = start + chunk_size
        probs = model.predict_proba(X_test.iloc[start:end]).astype(np.float32)
        top_k = np.argsort(probs, axis=1)[:, -k:]
        batch_y = y_test[start:end]
        correct += sum(batch_y[i] in top_k[i] for i in range(len(batch_y)))
    return correct / len(X_test)

top20_acc = chunked_top_k_accuracy(model, X_test, y_test, k=20)
print(f"Top-20 test accuracy: {top20_acc:.4f}")


705 species kept (down from 2936), 626116 rows
190.3s for 10 rounds -> roughly 5709s for 300 rounds
Training until validation scores don't improve for 20 rounds
Early stopping, best iteration is:
[262]	valid_0's multi_logloss: 4.96723
Stopped at round: 262
Train accuracy: 0.1295
Test accuracy:  0.0746
Top-20 test accuracy: 0.4455


In [ ]:
joblib.dump(model, "gb_model.pkl")

joblib.dump(label_encoder, "gb_label_encoder.pkl")

['label_encoderGB.pkl']

In [ ]:
cat_categories = {col: df_viable[col].cat.categories.tolist() for col in cat_features}
joblib.dump(cat_categories, "gb_cat_categories.pkl")

['cat_categories.pkl']